In [1]:
import os
import sys

from getpass import getpass

token = getpass("GitHub PAT: ")
os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/Hanh3103/BA_USDependency.git

%cd /content/BA_USDependency

!git checkout Tool

current_dir = os.getcwd()
repo_root = current_dir
while not os.path.exists(os.path.join(repo_root, "app")):
    parent = os.path.dirname(repo_root)
    if parent == repo_root:  # Reached system root without finding it
        repo_root = os.path.abspath(os.path.join(current_dir, "..", "..", "..")) 
        break
    repo_root = parent

app_path = os.path.abspath("app")
if app_path not in sys.path:
    sys.path.insert(0, app_path)



GitHub PAT: ··········
Cloning into 'BA_USDependency'...
remote: Enumerating objects: 986, done.
remote: Counting objects: 100% (386/386), done.
remote: Compressing objects: 100% (265/265), done.
remote: Total 986 (delta 181), reused 289 (delta 101), pack-reused 600 (from 1)
Receiving objects: 100% (986/986), 2.14 MiB | 26.11 MiB/s, done.
Resolving deltas: 100% (463/463), done.
/content/BA_USDependency
Branch 'Tool' set up to track remote branch 'Tool' from 'origin'.
Switched to a new branch 'Tool'


In [2]:
print("Repo Root:", repo_root)
print("App Path added to sys.path:", app_path)

Repo Root: /content/BA_USDependency
App Path added to sys.path: /content/BA_USDependency/app


In [7]:
# 1. Install dependencies
!pip install -r app/requirements.txt

# 2. Re-create the .env file with required settings
env_content = """OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_API_KEY=ollama
OLLAMA_MODELS=qwen3:8b,llama3.2:3b
OLLAMA_TEMPERATURE=0.0
OLLAMA_TOP_P=0.1
OLLAMA_NUM_PREDICT=2048
OLLAMA_NUM_CTX=8192
OLLAMA_KEEP_ALIVE=20m
OLLAMA_THINK=false
OLLAMA_FORMAT=json
"""

with open(".env", "w") as f:
    f.write(env_content.strip())

print("Dependencies installed and .env created.")

Dependencies installed and .env created.


In [4]:
!apt-get update -qq
!apt-get install -y zstd

# 1. Install Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Launch background service with GPU and parallel execution settings
import subprocess
import time

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_KEEP_ALIVE"] = "-1"      # Keep models pinned in VRAM
env["OLLAMA_NUM_PARALLEL"] = "4"     # Enable parallel pipeline processing

ollama_process = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)  # Wait for server startup

# 3. Pull required models
!ollama pull llama3.1:8b
!ollama pull qwen3:8b

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 76 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (295 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-l

In [8]:
# Verify GPU availability in Ollama
!nvidia-smi

# Test importing settings (should succeed without ValidationError)
from config import settings

print("Base URL:", settings.ollama_base_url)
print("Environment setup complete and ready to execute!")

Wed Aug 26 11:19:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
import re
import json
from itertools import combinations
import pandas as pd

In [10]:
from config import settings
from stages.dependency_pipeline_text import run_dependency_pipeline_text
from stages.dependency_pipeline_annotated import run_dependency_pipeline

qwen3:8b


In [11]:
GT_PATH = os.path.join("eval_with_colab","dependency_analysis","baseline", "Ground_truth_OBD_dependency.xlsx")

if os.path.exists(GT_PATH):
    print("Ground truth file found successfully!")
else:
    print("Error: Ground truth file not found. Check path.")

GT_SHEET_NAME = "OBD_Matrix"
DIRECTED_GT = True
ROW_IS_PREREQUISITE = False
ID_WIDTH = 3


DATASET_PATH_GT = os.path.join("eval_with_colab","annotation","baseline", "20US_baseline.json")
if os.path.exists(DATASET_PATH_GT):
    print("Annotated story file found successfully!")
else:
    print("Error: Annotated story  file not found. Check path.")

Ground truth file found successfully!
Annotated story file found successfully!


In [ ]:
DATASET_PATH_LLM_ANNOTATED = os.path.join("eval_with_colab", "annotation", "prediction", "20US_llama_annotated.json")

if os.path.exists(DATASET_PATH_LLM_ANNOTATED):
    print("LLM-annotated story file found successfully!")
else:
    print("Error: LLM-annotated story file not found. Check path.")

In [12]:
MODELS = ["llama3.1:8b", "qwen3:8b"]

In [13]:
COLOR_MAP = {
    "TP": "background-color: #6aa84f; color: white",
    "TP (dir X)": "background-color: #e69138; color: white",
    "FP": "background-color: #cc4125; color: white",
    "FN": "background-color: #f1c232; color: black",
    "-": "background-color: #d9d9d9; color: #666666",
}

## Helper functions

**Load ground truth from excel**

In [14]:
def normalize_id(raw) -> str:
    """Turn 'US_1', 'US-1', 'US-001', 1, etc. into a canonical 'US-001' form."""
    s = str(raw).strip()
    m = re.search(r"(\d+)", s)
    if not m:
        raise ValueError(f"Could not extract a story number from id: {raw!r}")
    num = int(m.group(1))
    return f"US-{num:0{ID_WIDTH}d}"

In [15]:
def _detect_header_row(path: str, sheet_name: str, corner_label: str = "US_i \\ US_j") -> int:
    """The GT sheet has a title + description row above the real header, and the
    description row happens to contain the substrings 'US_i'/'US_j' too -- so we
    require an EXACT match on the corner cell to find the true header row."""
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    for i, val in raw.iloc[:, 0].items():
        if isinstance(val, str) and val.strip() == corner_label:
            return i
    raise ValueError(
        f"Could not find header row with corner label {corner_label!r} in sheet {sheet_name!r}. "
        f"Inspect the sheet manually with pd.read_excel(path, sheet_name=sheet_name, header=None)."
    )

In [16]:
def load_ground_truth(path: str):
    """Read the GT matrix sheet and return:
    - gt_pairs: set of frozenset({id_i, id_j}) -- undirected, for TP/FP/FN scoring
    - gt_directed: set of (prerequisite_id, dependent_id) tuples -- for direction scoring
    - full_ids: every story ID present in the GT sheet, in original row order
    """
    header_row_idx = _detect_header_row(path, GT_SHEET_NAME)
    df = pd.read_excel(path, sheet_name=GT_SHEET_NAME, header=header_row_idx, index_col=0)

    gt_pairs, gt_directed = set(), set()
    full_ids = [normalize_id(x) for x in df.index]

    for row_label in df.index:
        for col_label in df.columns:
            val = df.loc[row_label, col_label]
            if pd.isna(val):
                continue
            val_str = str(val).strip()
            if val_str in ("1", "1.0") or val is True:
                try:
                    i = normalize_id(row_label)
                    j = normalize_id(col_label)
                except ValueError:
                    continue
                if i == j:
                    continue
                gt_pairs.add(frozenset({i, j}))
                if DIRECTED_GT:
                    if ROW_IS_PREREQUISITE:
                        gt_directed.add((i, j))
                    else:
                        gt_directed.add((j, i))

    return gt_pairs, gt_directed, full_ids

In [17]:
gt_pairs, gt_directed, full_ids = load_ground_truth(GT_PATH)

print(f"Loaded {len(full_ids)} story IDs from GT sheet")
print(f"Found {len(gt_directed)} directed pairs (prerequisite -> dependent)")

Loaded 20 story IDs from GT sheet
Found 10 directed pairs (prerequisite -> dependent)


In [18]:
print(gt_directed)

{('US-009', 'US-017'), ('US-001', 'US-002'), ('US-009', 'US-008'), ('US-005', 'US-017'), ('US-013', 'US-017'), ('US-005', 'US-008'), ('US-009', 'US-011'), ('US-005', 'US-014'), ('US-013', 'US-014'), ('US-005', 'US-011')}


**Evaluation loop**

In [19]:
import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Type
import requests
from pydantic import BaseModel

from config import settings
from schemas.dependency.dependency_schema import AnnotatedDependencyResult, TextDependencyResult

In [33]:
FORCE_RERUN = False
CONDITION_NAMES = ["text_gt", "annotated_gt", "annotated_llm"]

RESULTS_DIR = Path(repo_root) / "eval_with_colab" / "dependency_analysis" / "eval_outputs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ERRORS_FILE = RESULTS_DIR / "errors.json"
print(f"Results will be saved to {RESULTS_DIR}")
print(f"Errors will be saved to {ERRORS_FILE}")

Results will be saved to /content/BA_USDependency/eval_with_colab/dependency_analysis/eval_outputs
Errors will be saved to /content/BA_USDependency/eval_with_colab/dependency_analysis/eval_outputs/errors.json


In [22]:
def unload_ollama_model(model_name: str) -> None:
    """Evict current model from GPU memory to prevent VRAM OOM on model switch."""
    try:
        requests.post(
            f"{settings.ollama_base_url}/api/generate",
            json={"model": model_name, "keep_alive": 0},
            timeout=5,
        )
    except Exception:
        pass

In [34]:
def resolve_condition_config(condition_name: str) -> Tuple[Path, Any, Type[BaseModel]]:
    """Maps evaluation condition to dataset path, pipeline function, and Pydantic schema."""
    if condition_name == "text_gt":
        return DATASET_PATH_GT, run_dependency_pipeline_text, TextDependencyResult
    elif condition_name == "annotated_gt":
        return DATASET_PATH_GT, run_dependency_pipeline, AnnotatedDependencyResult
    elif condition_name == "annotated_llm":
        return DATASET_PATH_LLM_ANNOTATED, run_dependency_pipeline, AnnotatedDependencyResult
    else:
        raise ValueError(f"Unknown condition: {condition_name}")

In [24]:
def load_cached_results(
    results_file: Path, meta_file: Path, schema: Type[BaseModel]
) -> Tuple[List[BaseModel], float]:
    """Reads cached predictions and timing metadata from disk."""
    meta_data = json.loads(meta_file.read_text(encoding="utf-8"))
    raw_dicts = json.loads(results_file.read_text(encoding="utf-8"))
    results = [schema.model_validate(d) for d in raw_dicts]
    return results, meta_data["runtime_seconds"]

In [25]:
def save_results_to_cache(
    results_file: Path, meta_file: Path, results: List[BaseModel], runtime_seconds: float
) -> None:
    """Serializes execution outputs and timing metadata to disk."""
    result_dicts = [r.model_dump() for r in results]
    results_file.write_text(json.dumps(result_dicts, indent=2), encoding="utf-8")

    n_pairs = len(results)
    meta_data = {
        "runtime_seconds": runtime_seconds,
        "n_pairs": n_pairs,
        "seconds_per_pair": (runtime_seconds / n_pairs) if n_pairs > 0 else None,
    }
    meta_file.write_text(json.dumps(meta_data, indent=2), encoding="utf-8")

In [26]:
def save_errors_to_disk(errors: List[Dict[str, Any]]) -> None:
    """Persists error logs instantly so failures survive notebook runtime disconnects."""
    if errors:
        ERRORS_FILE.write_text(json.dumps(errors, indent=2), encoding="utf-8")

In [31]:
def run_single_condition(model_name: str, condition_name: str):
    """Executes a single pipeline condition for a specific model."""
    # 1. Handle VRAM eviction and update active model
    if settings.models and settings.models[0] != model_name:
        unload_ollama_model(settings.models[0])
    settings.models[0] = model_name

    # 2. Resolve configuration mapping
    dataset_path, pipeline_fn, schema_class = resolve_condition_config(condition_name)

    if not Path(dataset_path).exists():
        raise FileNotFoundError(f"Dataset path does not exist: {dataset_path}")

    print(f"== Running {model_name} | Condition: '{condition_name}' ==")

    # 3. Execute pipeline
    start_time = time.perf_counter()
    results, kept_ids = pipeline_fn(data_set_path=dataset_path)
    runtime_seconds = time.perf_counter() - start_time

    print(f"Finished {len(results)} pairs in {runtime_seconds:.1f} seconds.\n")
    return results, runtime_seconds, kept_ids

**LLAMA-EXECUTION**

In [32]:
llama_text_res, llama_text_time, llama_text_ids = run_single_condition(
    model_name="llama3.1:8b",
    condition_name="text_gt"
)

llama_ann_corr_res, llama_ann_corr_time, llama_ann_corr_ids = run_single_condition(
    model_name="llama3.1:8b",
    condition_name="annotated_gt"
)

== Running llama3.1:8b | Condition: 'text' ==
Kept 16 well-formed stories, dropped 4 ill-formed stories.
Analyzing pair: US-002 and US-003
{"story_a_id": "US-002", "story_b_id": "US-003", "dependencyFound": false, "dependencies": []}
Analyzing pair: US-002 and US-004
{"story_a_id": "US-002", "story_b_id": "US-004", "dependencyFound": false, "dependencies": []}
Analyzing pair: US-002 and US-005
{"story_a_id": "US-002", "story_b_id": "US-005", "dependencyFound": false, "dependencies": []}
Analyzing pair: US-002 and US-006
{"story_a_id": "US-002", "story_b_id": "US-006", "dependencyFound": false, "dependencies": []}
Analyzing pair: US-002 and US-007
{"story_a_id": "US-002", "story_b_id": "US-007", "dependencyFound": false, "dependencies": []}
Analyzing pair: US-002 and US-009
{"story_a_id": "US-002", "story_b_id": "US-009", "dependencyFound": false, "dependencies": []}
Analyzing pair: US-002 and US-010
{"story_a_id": "US-002", "story_b_id": "US-010", "dependencyFound": false, "dependencie

**QWEN-EXECUTION**

In [ ]:
qwen_text_res, qwen_text_time, qwen_text_ids = run_single_condition(
    model_name="qwen3:8b",
    condition_name="text_gt"
)

qwen_ann_corr_res, qwen_ann_corr_time, qwen_ann_corr_ids = run_single_condition(
    model_name="qwen3:8b",
    condition_name="annotated_gt"
)

**Evaluation with Direction**

In [ ]:
def _compute_prf1(tp: int, fp: int, fn: int) -> Dict[str, Any]:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp, "fp": fp, "fn": fn
    }

In [ ]:
from typing import Set, Tuple, Dict, Any, List

In [ ]:
def extract_predicted_directed(results: List[Any], use_llm_decision: bool = True) -> Set[Tuple[str, str]]:
    """Automatically detects schema type and extracts dependency with their direction tuples."""
    pred_directed: Set[Tuple[str, str]] = set()

    for item in results:
        data = item.model_dump() if hasattr(item, "model_dump") else item

        # Case A: Annotated condition schema (has final_llm or final_nlp)
        if "final_llm" in data or "final_nlp" in data:
            found_key = "dependencyFoundLLM" if use_llm_decision else "dependencyFoundNLP"
            if not data.get(found_key, True):
                continue

            key = "final_llm" if use_llm_decision else "final_nlp"
            final_block = data.get(key, {})
            dependencies = final_block.get("dependencies", [])

        # Case B: Text condition schema (has dependencyFound and dependencies list)
        elif data.get("dependencyFound", False):
            dependencies = data.get("dependencies", [])
        else:
            dependencies = []

        # Extract directed pairs from dependencies
        for dep in dependencies:
            prereq = str(dep.get("prerequisite_id")).strip()
            dependent = str(dep.get("dependent_id")).strip()
            if prereq and dependent and prereq != dependent:
                pred_directed.add((prereq, dependent))

    return pred_directed

In [ ]:
def evaluate_pipeline_run(
    results: List[Any],
    gt_directed: Set[Tuple[str, str]],
    valid_ids: Set[str],
    use_llm_decision: bool = True
) -> Dict[str, Any]:
    """Unified evaluation function for any pipeline condition."""
    filtered_gt_dir = {(i, j) for (i, j) in gt_directed if i in valid_ids and j in valid_ids}

    pred_directed = extract_predicted_directed(results, use_llm_decision=use_llm_decision)

    tp = len(pred_directed & filtered_gt_dir)
    fp = len(pred_directed - filtered_gt_dir)
    fn = len(filtered_gt_dir - pred_directed)

    return _compute_prf1(tp, fp, fn)

In [ ]:
def evaluate_pipeline_run_undirected(results, gt_pairs, valid_ids, use_llm_decision=True) -> Dict[str, Any]:
    """Direction-agnostic: was the dependency found at all, regardless of which side got called prerequisite vs. dependent."""
    filtered_gt = set()
    for pair in gt_pairs:
        i, j = tuple(pair)
        if i in valid_ids and j in valid_ids:
            filtered_gt.add(pair)

    pred_directed = extract_predicted_directed(results, use_llm_decision=use_llm_decision)
    pred_pairs = {frozenset(t) for t in pred_directed}
    tp = len(pred_pairs & filtered_gt)
    fp = len(pred_pairs - filtered_gt)
    fn = len(filtered_gt - pred_pairs)
    return _compute_prf1(tp, fp, fn)

In [ ]:
def evaluate_direction_errors(results, gt_directed, valid_ids, use_llm_decision=True) -> List[Tuple[str, str]]:
    """Pairs where the dependency was correctly found but prerequisite/dependent
    got swapped -- your COLOR_MAP's 'TP (dir X)' case, made explicit."""
    filtered_gt_dir = {(i, j) for (i, j) in gt_directed if i in valid_ids and j in valid_ids}
    pred_directed = extract_predicted_directed(results, use_llm_decision=use_llm_decision)
    reversed_gt = {(j, i) for (i, j) in filtered_gt_dir}
    return sorted(pred_directed & reversed_gt)

In [ ]:
# 1. Evaluate Text Condition
llama_text_metrics = evaluate_pipeline_run(llama_text_res, gt_directed, llama_text_ids)
llama_text_undirected = evaluate_pipeline_run_undirected(llama_text_res, gt_pairs, llama_text_ids)
llama_text_dir_errors = evaluate_direction_errors(llama_text_res, gt_directed, llama_text_ids)

# 2. Evaluate Annotated Condition (reusing the exact same function)
llama_annotated_metrics = evaluate_pipeline_run(llama_ann_corr_res, gt_directed, llama_ann_corr_ids, use_llm_decision=True)
llama_annotated_undirected = evaluate_pipeline_run_undirected(llama_ann_corr_res, gt_pairs, llama_ann_corr_ids)
llama_annotated_dir_errors = evaluate_direction_errors(llama_ann_corr_res, gt_directed, llama_ann_corr_ids)


In [ ]:
qwen_text_metrics = evaluate_pipeline_run(qwen_text_res, gt_directed, qwen_text_ids)
qwen_annotated_metrics = evaluate_pipeline_run(qwen_ann_corr_res, gt_directed, qwen_ann_corr_ids, use_llm_decision=True)

qwen_text_undirected = evaluate_pipeline_run_undirected(qwen_text_res, gt_pairs, qwen_text_ids)
qwen_annotated_undirected = evaluate_pipeline_run_undirected(qwen_ann_corr_res, gt_pairs, qwen_ann_corr_ids)

qwen_text_dir_errors = evaluate_direction_errors(qwen_text_res, gt_directed, qwen_text_ids)
qwen_annotated_dir_errors = evaluate_direction_errors(qwen_ann_corr_res, gt_directed, qwen_ann_corr_ids)

In [ ]:
summary_data = [
    {
        "Model": "llama3.1:8b", "Condition": "Text Prompt",
        "Precision (dir)": ["precision"], "Recall (dir)": ["recall"],
        "F1 (dir)": ["f1"], "F1 (undirected)": llama_text_undirected["f1"],
        "Direction errors": len(llama_text_dir_errors), "Time (s)": round(llama_text_time, 2),
        "TP": ["tp"], "FP": ["fp"], "FN": ["fn"],
    },
    {
        "Model": "llama3.1:8b", "Condition": "Annotated (CRUD) Prompt",
        "Precision (dir)": llama_annotated_metrics["precision"], "Recall (dir)": llama_annotated_metrics["recall"],
        "F1 (dir)": llama_annotated_metrics["f1"], "F1 (undirected)": llama_annotated_undirected["f1"],
        "Direction errors": len(llama_annotated_dir_errors), "Time (s)": round(llama_ann_corr_time, 2),
        "TP": llama_annotated_metrics["tp"], "FP": llama_annotated_metrics["fp"], "FN": llama_annotated_metrics["fn"],
    },
    {
        "Model": "qwen3:8b", "Condition": "Text Prompt",
        "Precision (dir)": qwen_text_metrics["precision"], "Recall (dir)": qwen_text_metrics["recall"],
        "F1 (dir)": qwen_text_metrics["f1"], "F1 (undirected)": qwen_text_undirected["f1"],
        "Direction errors": len(qwen_text_dir_errors), "Time (s)": round(qwen_text_time, 2),
        "TP": qwen_text_metrics["tp"], "FP": qwen_text_metrics["fp"], "FN": qwen_text_metrics["fn"],
    },
    {
        "Model": "qwen3:8b", "Condition": "Annotated (CRUD) Prompt",
        "Precision (dir)": qwen_annotated_metrics["precision"], "Recall (dir)": qwen_annotated_metrics["recall"],
        "F1 (dir)": qwen_annotated_metrics["f1"], "F1 (undirected)": qwen_annotated_undirected["f1"],
        "Direction errors": len(qwen_annotated_dir_errors), "Time (s)": round(qwen_ann_corr_time, 2),
        "TP": qwen_annotated_metrics["tp"], "FP": qwen_annotated_metrics["fp"], "FN": qwen_annotated_metrics["fn"],
    },
]

df_summary = pd.DataFrame(summary_data)
display(df_summary)
df_summary.to_csv(RESULTS_DIR / "rq1_summary_all_models.csv", index=False)

In [ ]:
if ERRORS_FILE.exists():
    errors = json.loads(ERRORS_FILE.read_text(encoding="utf-8"))
    print(f"{len(errors)} pairs failed during processing:")
    for e in errors:
        print(e)
else:
    print("No errors logged.")

In [ ]:
latex_table = df_summary.to_latex(index=False, float_format="%.4f",
                                    caption="RQ1: Text-based vs. CRUD-annotated dependency detection across models",
                                    label="tab:rq1_summary")
(RESULTS_DIR / "rq1_summary.tex").write_text(latex_table, encoding="utf-8")
print(latex_table)

In [ ]:
conditions = [
    ("llama3.1:8b", "text_gt", llama_text_res, llama_text_time),
    ("llama3.1:8b", "annotated_gt", llama_ann_corr_res, llama_ann_corr_time),
    ("qwen3:8b", "text_gt", qwen_text_res, qwen_text_time),
    ("qwen3:8b", "annotated_gt", qwen_ann_corr_res, qwen_ann_corr_time),
]

for model_name, condition_name, results, runtime in conditions:
    safe_model = model_name.replace(":", "_")
    results_file = RESULTS_DIR / f"{safe_model}_{condition_name}_results.json"
    meta_file = RESULTS_DIR / f"{safe_model}_{condition_name}_meta.json"
    save_results_to_cache(results_file, meta_file, results, runtime)
    print(f"Cached: {results_file.name}")